In [16]:
import sys, json, re, warnings
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from langdetect import detect

warnings.filterwarnings("ignore")
sys.path.append(str(Path.cwd().parent / "src"))
from config import (
    PROJECT_ROOT, DATA_RAW, DATA_PROCESSED,
    REDDIT_RAW_FILE, TWITTER_RAW_FILE,
    REDDIT_COLUMNS, TWITTER_COLUMNS,
    PRIORITY_SUBREDDITS, EVENT_START, TIME_WINDOW_HOURS, RANDOM_SEED
)
np.random.seed(RANDOM_SEED)

nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("punkt_tab", quiet=True)

print("✅ Config loaded")
print("Reddit file :", REDDIT_RAW_FILE)
print("Twitter file:", TWITTER_RAW_FILE)

✅ Config loaded
Reddit file : C:\Users\GALLOP SEA\Desktop\The_Echo_Of_Anonimity-1\data\raw\reddit_israel_hamas.csv
Twitter file: C:\Users\GALLOP SEA\Desktop\The_Echo_Of_Anonimity-1\data\raw\X_israel_hamas_r.csv


In [17]:
reddit_raw = pd.read_csv(REDDIT_RAW_FILE, low_memory=False)
print("Reddit raw shape:", reddit_raw.shape)
print("Columns:", reddit_raw.columns.tolist())
reddit_raw.head(2)

Reddit raw shape: (10234, 25)
Columns: ['comment_id', 'score', 'self_text', 'subreddit', 'created_time', 'post_id', 'author_name', 'controversiality', 'ups', 'downs', 'user_is_verified', 'user_account_created_time', 'user_awardee_karma', 'user_awarder_karma', 'user_link_karma', 'user_comment_karma', 'user_total_karma', 'post_score', 'post_self_text', 'post_title', 'post_upvote_ratio', 'post_thumbs_ups', 'post_total_awards_received', 'post_created_time', 'account_age_days']


,comment_id,score,self_text,subreddit,created_time,post_id,author_name,controversiality,ups,downs,...,user_comment_karma,user_total_karma,post_score,post_self_text,post_title,post_upvote_ratio,post_thumbs_ups,post_total_awards_received,post_created_time,account_age_days
0,k6stxkz,534,You haven't been taught since preschool to mur...,combatfootage,2023-10-28 10:04:42,17i973e,DdCno1,0,534,0,...,525710.0,564489.0,10418,NaN,Al Bireh this morning 28-02 .A Palestinian you...,0.94,10418,0,2023-10-28 08:18:29,4438.0
1,k5wj9rq,330,The thing is when you're black it is easier to...,publicfreakout,2023-10-22 00:16:48,17dcapb,CantStopPoppin,0,330,0,...,823027.0,9874145.0,11953,NaN,Black Mother seeks answers after 5-year-old da...,0.93,11953,0,2023-10-21 21:00:16,2380.0


In [18]:
twitter_raw = pd.read_csv(TWITTER_RAW_FILE, low_memory=False)
print("Twitter raw shape:", twitter_raw.shape)
print("Columns:", twitter_raw.columns.tolist())
twitter_raw.head(2)

Twitter raw shape: (12000, 10)
Columns: ['user', 'tweet_pub_time', 'tweet_text', 'link', 'tweet_likes', 'tweet_replies', 'tweet_reposts', 'tweets_num', 'following_num', 'followers_num']


,user,tweet_pub_time,tweet_text,link,tweet_likes,tweet_replies,tweet_reposts,tweets_num,following_num,followers_num
0,Babylon Bulletin,10/7/2023 0:14,An American tourist was detained in Jerusalem ...,https://twitter.com/BabylonBulletin/status/171...,270,19,13,12925,20,127068
1,baha,10/7/2023 0:46,"Israel: Covid outbreak spreads rapidly. ""There...",https://twitter.com/bahabreaking/status/178496...,39,5,5,19941,1284,6185


In [19]:
reddit_df = reddit_raw.rename(columns=REDDIT_COLUMNS)

# keep only relevant columns
reddit_keep = [
    "comment_id", "self_text", "created_time", "author_name", "subreddit",
    "score", "ups", "downs", "controversiality", "post_id", "post_title",
    "user_is_verified", "user_account_created_time",
    "user_total_karma", "user_comment_karma",
    "post_score", "post_upvote_ratio", "account_age_days",
]
reddit_df = reddit_df[[c for c in reddit_keep if c in reddit_df.columns]].copy()
print("Reddit standardized shape:", reddit_df.shape)
reddit_df.head(2)

Reddit standardized shape: (10234, 18)


,comment_id,self_text,created_time,author_name,subreddit,score,ups,downs,controversiality,post_id,post_title,user_is_verified,user_account_created_time,user_total_karma,user_comment_karma,post_score,post_upvote_ratio,account_age_days
0,k6stxkz,You haven't been taught since preschool to mur...,2023-10-28 10:04:42,DdCno1,combatfootage,534,534,0,0,17i973e,Al Bireh this morning 28-02 .A Palestinian you...,True,2011-09-03 01:07:26,564489.0,525710.0,10418,0.94,4438.0
1,k5wj9rq,The thing is when you're black it is easier to...,2023-10-22 00:16:48,CantStopPoppin,publicfreakout,330,330,0,0,17dcapb,Black Mother seeks answers after 5-year-old da...,True,2017-04-15 13:37:28,9874145.0,823027.0,11953,0.93,2380.0


In [21]:
twitter_df = twitter_raw.rename(columns=TWITTER_COLUMNS)

twitter_keep = [
    "user", "tweet_pub_time", "tweet_text", "link",
    "tweet_likes", "tweet_replies", "tweet_reposts",
    "tweets_num", "following_num", "followers_num",
]
twitter_df = twitter_df[[c for c in twitter_keep if c in twitter_df.columns]].copy()
print("Twitter standardized shape:", twitter_df.shape)
twitter_df.head(2)

Twitter standardized shape: (12000, 10)


,user,tweet_pub_time,tweet_text,link,tweet_likes,tweet_replies,tweet_reposts,tweets_num,following_num,followers_num
0,Babylon Bulletin,10/7/2023 0:14,An American tourist was detained in Jerusalem ...,https://twitter.com/BabylonBulletin/status/171...,270,19,13,12925,20,127068
1,baha,10/7/2023 0:46,"Israel: Covid outbreak spreads rapidly. ""There...",https://twitter.com/bahabreaking/status/178496...,39,5,5,19941,1284,6185


In [22]:
reddit_df["subreddit_lc"] = reddit_df["subreddit"].astype(str).str.lower()
before = len(reddit_df)
# reddit_df = reddit_df[reddit_df["subreddit_lc"].isin({s.lower() for s in PRIORITY_SUBREDDITS})]
reddit_df = reddit_df.drop(columns=["subreddit_lc"])
print(f"Subreddit filter: {before:,} -> {len(reddit_df):,}")

Subreddit filter: 10,234 -> 10,234


In [23]:
# Reddit
reddit_df["timestamp"] = pd.to_datetime(reddit_df["created_time"], errors="coerce", utc=True)
reddit_df = reddit_df.dropna(subset=["timestamp"])
reddit_df["timestamp"] = reddit_df["timestamp"].dt.tz_localize(None)

# Twitter
twitter_df["timestamp"] = pd.to_datetime(twitter_df["tweet_pub_time"], errors="coerce", utc=True)
twitter_df = twitter_df.dropna(subset=["timestamp"])
twitter_df["timestamp"] = twitter_df["timestamp"].dt.tz_localize(None)

print("Reddit range :", reddit_df["timestamp"].min(), "->", reddit_df["timestamp"].max())
print("Twitter range:", twitter_df["timestamp"].min(), "->", twitter_df["timestamp"].max())

Reddit range : 2023-10-07 01:04:22 -> 2023-10-30 23:53:36
Twitter range: 2023-10-07 00:14:00 -> 2023-10-30 23:51:00


In [24]:
def add_temporal(df):
    df["year"]        = df["timestamp"].dt.year
    df["month"]       = df["timestamp"].dt.month
    df["day"]         = df["timestamp"].dt.day
    df["hour"]        = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.dayofweek
    df["is_weekend"]  = df["day_of_week"].isin([5, 6])
    df["hours_since_event"] = (df["timestamp"] - EVENT_START).dt.total_seconds() / 3600
    df["hour_bin"] = df["hours_since_event"].apply(lambda x: int(x) if x >= 0 else -1)
    return df

reddit_df  = add_temporal(reddit_df)
twitter_df = add_temporal(twitter_df)

# keep only rows within window
reddit_df  = reddit_df[(reddit_df["hours_since_event"] >= 0) &
                       (reddit_df["hours_since_event"] <= TIME_WINDOW_HOURS)].copy()
twitter_df = twitter_df[(twitter_df["hours_since_event"] >= 0) &
                        (twitter_df["hours_since_event"] <= TIME_WINDOW_HOURS)].copy()

print("After window — Reddit:", len(reddit_df), " Twitter:", len(twitter_df))

After window — Reddit: 10234  Twitter: 12000


In [25]:
STOP = set(stopwords.words("english"))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    t = text.lower()
    t = re.sub(r"http\S+|www\S+|https\S+", "[URL]", t)
    t = re.sub(r"\[deleted\]|\[removed\]", "", t)
    t = re.sub(r"u/\w+", "[USER]", t)
    t = re.sub(r"r/\w+", "[SUBREDDIT]", t)
    t = re.sub(r"@\w+", "[USER]", t)
    t = re.sub(r"#", "", t)
    t = re.sub(r"[^a-zA-Z0-9\s\.\,\?\!\'\"-]", " ", t)
    return " ".join(t.split())

def tokenize(text):
    if not text:
        return []
    try:
        return [w for w in word_tokenize(text)
                if w.lower() not in STOP and len(w) > 2]
    except Exception:
        return []

for df, col in [(reddit_df, "self_text"), (twitter_df, "tweet_text")]:
    df["clean_text"] = df[col].apply(clean_text)
    df["tokens"]     = df["clean_text"].apply(tokenize)
    df["word_count"] = df["tokens"].apply(len)

# drop very short comments
reddit_df  = reddit_df[reddit_df["word_count"] >= 5].copy()
twitter_df = twitter_df[twitter_df["word_count"] >= 5].copy()
print("After cleaning — Reddit:", len(reddit_df), " Twitter:", len(twitter_df))

After cleaning — Reddit: 9447  Twitter: 11785


In [26]:
def is_en(text):
    if not text or len(text) < 10:
        return False
    try:
        return detect(text) == "en"
    except Exception:
        return False

reddit_df["language"]  = reddit_df["clean_text"].apply(lambda x: "en" if is_en(x) else "other")
twitter_df["language"] = twitter_df["clean_text"].apply(lambda x: "en" if is_en(x) else "other")

reddit_df  = reddit_df[reddit_df["language"] == "en"].copy()
twitter_df = twitter_df[twitter_df["language"] == "en"].copy()
print("After English filter — Reddit:", len(reddit_df), " Twitter:", len(twitter_df))

After English filter — Reddit: 9406  Twitter: 11729


In [27]:
reddit_df["platform"]  = "Reddit"
twitter_df["platform"] = "X"

# Reddit engagement
reddit_df["engagement_raw"] = reddit_df["score"].fillna(0)
reddit_df["engagement_log"] = np.log1p(reddit_df["engagement_raw"].clip(lower=0))
mx = reddit_df["engagement_log"].max() or 1
reddit_df["engagement_normalized"] = reddit_df["engagement_log"] / mx

# Twitter engagement
tw = twitter_df
tw["engagement_raw"] = (
    tw.get("tweet_likes", 0).fillna(0)
    + 2 * tw.get("tweet_reposts", 0).fillna(0)
    + 0.5 * tw.get("tweet_replies", 0).fillna(0)
)
tw["engagement_log"] = np.log1p(tw["engagement_raw"].clip(lower=0))
mx = tw["engagement_log"].max() or 1
tw["engagement_normalized"] = tw["engagement_log"] / mx
twitter_df = tw

print("✅ Engagement computed")

✅ Engagement computed


In [28]:
reddit_df.to_parquet(DATA_PROCESSED / "reddit_processed.parquet", index=False)
twitter_df.to_parquet(DATA_PROCESSED / "twitter_processed.parquet", index=False)

summary = {
    "reddit_rows":  int(len(reddit_df)),
    "twitter_rows": int(len(twitter_df)),
    "event":        EVENT_NAME if "EVENT_NAME" in dir() else "Israel-Hamas War",
    "window_hours": TIME_WINDOW_HOURS,
}
with open(DATA_PROCESSED / "processing_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("✅ Saved. Reddit:", len(reddit_df), " Twitter:", len(twitter_df))
print(summary)

✅ Saved. Reddit: 9406  Twitter: 11729
{'reddit_rows': 9406, 'twitter_rows': 11729, 'event': 'Israel-Hamas War', 'window_hours': 1440}
